In [2]:
import mlflow

TRACKING_SERVER_HOST="ec2-44-201-232-211.compute-1.amazonaws.com"
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")

In [3]:
print(f"tracking_uri: '{mlflow.get_tracking_uri()}'")

tracking_uri: 'http://ec2-44-201-232-211.compute-1.amazonaws.com:5000'


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location='s3://mlflow-artifacts-remote-rashmin/0', creation_time=1774562029150, experiment_id='0', last_update_time=1774562029150, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("first_exp")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/03/27 03:37:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/27 03:37:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


default artifacts URI: 's3://mlflow-artifacts-remote-rashmin/1/ae2758082bfc4eae9ef782865e5e2b73/artifacts'
🏃 View run rumbling-snail-132 at: http://ec2-44-201-232-211.compute-1.amazonaws.com:5000/#/experiments/1/runs/ae2758082bfc4eae9ef782865e5e2b73
🧪 View experiment at: http://ec2-44-201-232-211.compute-1.amazonaws.com:5000/#/experiments/1


In [8]:
mlflow.search_experiments()

[<Experiment: artifact_location='s3://mlflow-artifacts-remote-rashmin/1', creation_time=1774562782504, experiment_id='1', last_update_time=1774562782504, lifecycle_stage='active', name='first_exp', tags={}, workspace='default'>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-rashmin/0', creation_time=1774562029150, experiment_id='0', last_update_time=1774562029150, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [9]:
from mlflow.tracking import MlflowClient

client=MlflowClient(f"http://{TRACKING_SERVER_HOST}:5000")

In [10]:
client.search_registered_models()

[]

In [11]:
run_id=client.search_runs(experiment_ids=['1'])[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris_classifier'
)

Successfully registered model 'iris_classifier'.
2026/03/27 03:44:54 WARNING mlflow.tracking._model_registry.fluent: Run with id ae2758082bfc4eae9ef782865e5e2b73 has no artifacts at artifact path 'models', registering model based on models:/m-09ef6ddc10944caa8248e3dc18d9dc68 instead
2026/03/27 03:44:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris_classifier, version 1
Created version '1' of model 'iris_classifier'.


<ModelVersion: aliases=[], creation_timestamp=1774563295374, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1774563295374, metrics=None, model_id=None, name='iris_classifier', params=None, run_id='ae2758082bfc4eae9ef782865e5e2b73', run_link='', source='models:/m-09ef6ddc10944caa8248e3dc18d9dc68', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>